<h1>Working with SFS data on Hopper</h1>
<h3>Aug. 2026</h3>

<h2>Introduction</h2>

This notebook demonstrates how to use the ```ufs-community/ufs-analysis``` repository located here: https://github.com/ufs-community/ufs-analysis.  If you are unfamiliar with either Git or Hopper, please refer to the README titled `Updating Github repositories, a Hopper-specific introduction` located here: https://github.com/cristianastan2/large-scale-dynamics.  Here, we assume that you will be using Python during your Hopper sessions.<br><br>

<h2>GMU Hopper Dashboard</h2>

Here is the GMU Hopper dashboard: https://ondemand.orc.gmu.edu/pun/sys/dashboard<br>
This is where you manage Hopper sessions.  For daily use, the most convenient way to use Python on Hopper is by launching a Jupyter session.<br><br>However, our first concern in this tutorial is creating the proper Python environment for ```ufs-analysis```.<br>***For all Python-packaging work, it is strongly recommended that you use HOPPER Shell Access.***<br>
In the top navigation panel, click the `Clusters` dropdown and then select `>_ HOPPER Shell Access`.  This will launch a Hopper session in a new window.  Using this session, we will install all necessary Python packages into our profile.<br>
<br>

<h2>Miniforge</h2>

First, we must install the right Python package manager (Miniforge).<br>
Instructions for doing so can be found here: https://wiki.orc.gmu.edu/mkdocs/Conda_Environments_on_Hopper/<br>

The two important instructions in that guide are to Download Miniforge and then Install Miniforge.  For this part, make sure you are in your HOME directory.  Then,<br>
Download Miniforge:<br>
```$ wget https://github.com/conda-forge/miniforge/releases/latest/download/Miniforge3-Linux-x86_64.sh```

Install Miniforge into your HOME directory:<br>
```$ bash Miniforge3-Linux-x86_64.sh -b -p $HOME/miniforge```


And lastly, source Miniforge:<br>
```$ source $HOME/miniforge2/bin/activate```<br>
Note that you must source Miniforge every time you start a new Hopper session.  Sourcing Miniforge will give you the `conda` command used later.
<br><br>

<h2>Clone repository</h2>

Now we can start working with the `ufs-analysis` repository.  Clone the repository into your HOME directory:<br>
```$ git clone git@github.com:ufs-community/ufs-analysis.git```<br>

Move into the `ufs-analysis` directory:<br>
```$ cd ufs-analysis```
<br><br>

<h2>Create new Python environment</h2>

Inside the `ufs-analysis` directory you'll see a file called `environment.yml`.  This file defines all the required packages and their versions needed to run `ufs-analysis` codes.<br><br>

We can create a new `Conda` environment (via Miniforge) by issuing the following command:<br>
```$ conda env create -f environment.yml```<br>

This will take a few minutes to complete.  After this is done, there will be a new Conda environment named `ufs_py311`.  See here:<br>
```$ conda env list```

```
# conda environments:
#
# ufs_py311              /home/<your_name>/miniforge2/envs/ufs_py311
```

Now you can activate environment like so:<br>
```$ conda activate ufs_py311```<br>

The active environment will prepend your terminal handle like so:<br>
```(ufs_py311) [<your_username>@hopper2 ~]$```<br>
Note that you will have to activate your environment every session.
<br><br>

<h2>Create Python kernel</h2>

To run interactive Jupyter notebooks in Hopper, you must first create a Python kernel for your environment.  The commands necessary to do so were already installed as part of the `ufs_py311` environment!<br><br>
To create a Python Kernel, make sure the environment you want to kernelize (`ufs_py311`) is active, then run:<br>
```$ ipython kernel install --name "ufs_py311" --user```<br>

To verify that your kernel was correctly installed, run:<br>
```jupyter kernelspec list```<br>

```
Available kernels:
  python3          /home/<your_username>/miniforge2/envs/ufs_py311/share/jupyter/kernels/python3
  ufs_py311        /home/<your_username>/.local/share/jupyter/kernels/ufs_py311
```
<br><br>

<h2>Interactive Jupyter Sessions</h2>

Now that the Python kernel is created, we can begin working with our code interactively via Jupyter.<br>Close the `>_ HOPPER Shell Access` and navigate back to the Hopper Dashboard.  Under `Servers` click on a Jupyter application instance (any one will do).  This will open a page with configurable options for your Jupyter session.  Under `Extra Arguments for Jupyter:` make sure you have something like `--notebook-dir=/home/<your_username>/`.  Then, based on your needs, you can configure `Time limit in hours:`, `Number of Cores:`, and `Memory GBs/core:`. You can probably always set `8 GBs/core`, but for time limit and number of cores, please be mindful not to reserve too many more resources than you need.  If you finish working before your session expires, then please manually close the session to free up computing resources for other users.<br><br>

Once you have decided on the parameters for your Jupyter session, click `Launch`.  A `Launcher` window will appear with buttons for opening a `Terminal` and `Jupyter` notebooks.  When you launch a Jupyter notebook, there is an indicator in the top-right of the screen for the Python kernel.  Make sure it is set to our newly created kernel, `ufs_py311`.

<h2>ufs-analysis</h2>

Let's start running code.<br>
The `ufs-analysis` package has 3 modules: `datareader`, `regridder`, and `util`.  This tutorial focuses on the first two modules.

<h2>DataReader class</h2>

The DataReader class is used to package model and verification data into a unifying, canonical data structure.  This class is the entry-point for all downstream functionality in the `ufs-analysis` package.  It offers three core pieces of functionality:<br>1. Querying data from remote file systems like Amazon S3 or Google Cloud Storage buckets,<br>2. Standardizing the coordinate system, and<br>3. Retrieving subsets of the data.<br><br>
One requirement is that datasets are assumed to exist as `zarr` files in their remote locations. Under the hood, the `ufs-analysis` package stores data as Xarray Datasets and DataArrays.<br><br>
Let's step through an example in which we query 1 UFS dataset and 1 ERA5 dataset.

<h2>DataReader example</h2>

First, you must define where the `ufs-analysis` source code is located.  The `basedir` variable should point to the root directory of the package.  For example, if this tutorial notebook is saved in the same directory as `ufs-analysis` itself, then set `basedir` as so:

In [1]:
basedir = './ufs-analysis'

In [2]:
import os
import sys

# Point to root directory of repository
root_dir = os.path.join(os.getcwd(), basedir)
if root_dir not in sys.path:
    sys.path.insert(0, root_dir)

# Import datareader module
from src.datareader import datareader as dr

As of Aug. 2026, the DataReader class has two built-in data source options: `UFS` and `ERA5`.  Programmatically speaking, these data sources are handled as sub-classes to the `DataReader` super class, with their only uniqueness being each particular implementation of Xarray's `open_zarr()` method.  Users need not be aware of these details, but the point here is that further subclasses can be easily defined to handle additional data sources (for example MERRA2) without interfering with the DataReader superclass, provided that a remote data source is indeed available for that model.

In [5]:
# Get SFS atmospheric model data
ufs_data_reader = dr.getDataReader(datasource='UFS',                  
                                   experiment = 'baseline',
                                   model='atm')

No filename provided; deferring to default
Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/baseline/atm_monthly.zarr


Here, the `experiment` parameter is special to UFS.  By default, a UFS-type DataReader pulls SFS experimental data from `s3://noaa-oar-sfsdev-pds/experiments/phase_1/<experiment>/atm_monthly.zarr`.<br>

If you navigate to the webpage for NOAA's S3 bucket here:<br>
`https://noaa-oar-sfsdev-pds.s3.amazonaws.com/index.html`
You'll see that `phase_1` has `baseline`, `beta.0.1`, `c96_beta.0.1`, and `cpc_ics` experiments are available.<br><br>This data bucket is subject to change as new experiments are undertaken and new data published or otherwise reorganized.  As such, an alternative option for specifying datasets is to use the `filename` parameter.  Here is an example of grabbing the same dataset using `filename`:


In [9]:
# Get SFS atmospheric model data
ufs_data_reader = dr.getDataReader(datasource='UFS',                  
                                   filename=f'experiments/phase_1/baseline/atm_monthly.zarr',
                                   model='atm')

Reading data from s3://noaa-oar-sfsdev-pds/experiments/phase_1/baseline/atm_monthly.zarr


For UFS, the base url is set to `s3://noaa-oar-sfsdev-pds/` onto which the `filename` parameter is joined.

The `model` parameter can be one of `atm`, `ocn`, `lnd`, `ice`, or `wav`.  As of Aug. 2026, this parameter is not truly active (i.e. does not actually do anything), but that will change as the `ufs-analysis` package is further developed for non-`atm` models.


Now let's get an ERA5 verification dataset:

In [10]:
era5_data_reader = dr.getDataReader(datasource='ERA5')

No filename provided; deferring to default
Reading data from gs://gcp-public-data-arco-era5/ar/1959-2022-6h-512x256_equiangular_conservative.zarr


The ERA5 DataReader has a base URL fixed to `gs://gcp-public-data-arco-era5/ar/`